# Markitdown Experiment

Experimenting with [MarkItDown](https://github.com/microsoft/markitdown) — Microsoft's utility that converts documents and images to Markdown for LLM pipelines.

Conversions covered, using files in `data/`:

| Section | Files | Converter |
|---|---|---|
| Doc | `doc/` | plain + `markitdown-ocr` plugin cell |
| Image | `images/` | plain (metadata) + LLM (description / transcription) |
| PDFs | `pdfs/` | plain text-layer extraction + scanned-PDF vision pipeline + OCR plugin |
| Scanned PDF | rendered PDF pages | PyMuPDF render → multimodal LLM (`gpt-6-luna`, reasoning effort `none`) |
| PPT | `powerpoints/` | plain + optional LLM alt-text + OCR plugin |
| Excel | `spreadsheets/` | plain (tables) + OCR plugin |

> Two LLM routes are used here:
> **1)** MarkItDown's built-in LLM support for **image files and pptx images** (`llm_client`/`llm_model` on the converter);
> **2)** the [`markitdown-ocr`](https://github.com/microsoft/markitdown#plugins) plugin, which OCRs images **embedded in pdf / docx / pptx / xlsx** via `MarkItDown(enable_plugins=True, ...)`.
> A scanned PDF has no text layer, so its pages are rendered with PyMuPDF and transcribed by the multimodal model directly — the plugin offers 300 DPI full-page fallback for the same scenario.

In [15]:
# Import libraries
import base64
import os
import time
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Markdown, display
from markitdown import MarkItDown
from openai import OpenAI
import pymupdf  # renders PDF pages to PNG (used for the scanned-PDF section)

DATA = Path("data")
OUT = Path("out")
OUT.mkdir(exist_ok=True)

print("data dir ok:", DATA.is_dir())

data dir ok: True


In [16]:
# Setup llm and keys
load_dotenv()  # reads OPENAI_API_KEY from .env

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("OPENAI_API_KEY is missing - add it to the .env file next to this notebook.")

client = OpenAI(api_key=api_key, max_retries=3)

# Multimodal model used for image / scanned-page transcription.
# Override in .env with OPENAI_MODEL=<name> if the default is unavailable.
MODEL = os.getenv("OPENAI_MODEL", "gpt-6-luna")
REASONING_EFFORT = "none"

# Shared transcription prompt for every LLM-powered conversion in this notebook.
OCR_PROMPT = (
    "Transcribe ALL text visible in this image into clean Markdown. "
    "Preserve headings with # markers, lists with -, and reproduce tables as "
    "GitHub-style Markdown tables. Keep the original language. Return only the Markdown."
)


class _CompletionsWithEffort:
    """Injects `reasoning_effort` into every chat.completions.create() call."""

    def __init__(self, completions, effort):
        self._completions = completions
        self._effort = effort

    def create(self, **kwargs):
        kwargs.setdefault("extra_body", {})
        kwargs["extra_body"]["reasoning_effort"] = self._effort
        return self._completions.create(**kwargs)

    def __getattr__(self, name):
        return getattr(self._completions, name)


class _ChatWithEffort:
    def __init__(self, chat, effort):
        self._chat = chat
        self.completions = _CompletionsWithEffort(chat.completions, effort)

    def __getattr__(self, name):
        return getattr(self._chat, name)


class ReasoningNoneClient:
    """Drop-in wrapper around an OpenAI client.

    MarkItDown and the markitdown-ocr plugin both call
    `llm_client.chat.completions.create(...)`, so hooking there forces
    reasoning_effort='none' for every conversion they perform.
    """

    def __init__(self, inner, effort="none"):
        self.chat = _ChatWithEffort(inner.chat, effort)
        self._inner = inner

    def __getattr__(self, name):
        return getattr(self._inner, name)


llm_client = ReasoningNoneClient(client, REASONING_EFFORT)

# 1) Plain converter: text/metadata extraction only, never calls the API.
md_plain = MarkItDown()
# 2) LLM converter: adds image descriptions / transcriptions via the multimodal model.
md_llm = MarkItDown(llm_client=llm_client, llm_model=MODEL)
# 3) Plugin converter: markitdown-ocr registers OCR-enhanced pdf/docx/pptx/xlsx
#    converters ahead of the built-ins when enable_plugins=True.
md_ocr = MarkItDown(
    enable_plugins=True,
    llm_client=llm_client,
    llm_model=MODEL,
    llm_prompt=OCR_PROMPT,
)

print("model:", MODEL, "| reasoning effort:", REASONING_EFFORT)
print("converters: md_plain (text only) | md_llm (vision images) | md_ocr (plugin OCR)")

model: gpt-6-luna | reasoning effort: none
converters: md_plain (text only) | md_llm (vision images) | md_ocr (plugin OCR)


In [17]:
# Shared helpers: run a conversion, preview it, save the full markdown
SUMMARY = []  # (source, converter kind, output chars)


def save_markdown(name, text):
    path = OUT / name
    path.write_text(text, encoding="utf-8")
    return path


def preview(text, chars=900):
    if len(text) <= chars:
        return text
    return text[:chars] + f"\n\n> ... {len(text) - chars:,} more characters (full text saved in `out/`)"


def convert_and_show(path, converter=md_plain, prompt=None, name=None, chars=900):
    """Convert a file, render a preview, and save the full markdown to out/."""
    path = Path(path)
    kwargs = {"llm_prompt": prompt} if prompt else {}
    result = converter.convert(str(path), **kwargs)
    text = result.markdown or ""
    kind = "ocr-plugin" if converter is md_ocr else ("llm" if converter is md_llm else "plain")
    out_path = save_markdown(name or (path.stem + ".md"), text)
    SUMMARY.append((str(path), kind, len(text)))
    display(Markdown(f"**`{path}`** -> `{out_path}` ({len(text):,} chars)\n\n---"))
    display(Markdown(preview(text, chars)))
    return text

## Doc

Word documents go through core preprocessing (styles, math, Mammoth HTML→Markdown), keeping headings, lists, and tables intact.

The `markitdown-ocr` plugin's enhanced DOCX converter OCRs embedded images and inserts each fragment inline where the image sat, wrapped as `*[Image OCR] … [End OCR]*`. The current article contains **no embedded images**, so its plugin run equals the plain output at zero API cost — flip the flag below once an image-bearing version is in place.

In [18]:
# Plain DOCX conversion
convert_and_show(DATA / "doc" / "multi_agent_workflows_ai_article.docx", chars=1100)

**`data\doc\multi_agent_workflows_ai_article.docx`** -> `out\multi_agent_workflows_ai_article.md` (9,108 chars)

---

**BEYOND A
SINGLE AI**

How Multi-Agent Workflows Turn Language Models into Collaborative Systems

*A practical introduction to agent roles, orchestration, shared context, and reliable execution*

**AI ENGINEERING FIELD NOTES**

# Introduction: From Chatbots to Coordinated Systems

A language model can summarize a report, write code, or answer a question. But many real-world tasks are not single-step problems. They require research, planning, execution, verification, and sometimes the ability to recover when something goes wrong. A multi-agent workflow organizes these activities into a coordinated process rather than asking one model to do everything at once.

The term “multi-agent” can sound more complicated than it needs to be. In practice, it often means assigning different responsibilities to separate model-driven components and defining how work moves between them. One component gathers evidence, another proposes a plan, a third performs a task, and a reviewer checks the result. The value comes not from the number of agents, but from the quality of the workflow.

# 1. What Is a 

> ... 8,008 more characters (full text saved in `out/`)

"**BEYOND A\nSINGLE AI**\n\nHow Multi-Agent Workflows Turn Language Models into Collaborative Systems\n\n*A practical introduction to agent roles, orchestration, shared context, and reliable execution*\n\n**AI ENGINEERING FIELD NOTES**\n\n# Introduction: From Chatbots to Coordinated Systems\n\nA language model can summarize a report, write code, or answer a question. But many real-world tasks are not single-step problems. They require research, planning, execution, verification, and sometimes the ability to recover when something goes wrong. A multi-agent workflow organizes these activities into a coordinated process rather than asking one model to do everything at once.\n\nThe term “multi-agent” can sound more complicated than it needs to be. In practice, it often means assigning different responsibilities to separate model-driven components and defining how work moves between them. One component gathers evidence, another proposes a plan, a third performs a task, and a reviewer checks

In [ ]:
# Plugin OCR: one vision call per unique embedded image (0 today; costs appear
# once you swap in an image-bearing docx). Set True to run the plugin path.
RUN_DOC_PLUGIN_OCR = False
if RUN_DOC_PLUGIN_OCR:
    convert_and_show(
        DATA / "doc" / "multi_agent_workflows_ai_article.docx",
        converter=md_ocr,
        name="ocr_multi_agent_workflows_ai_article.md",
        chars=1400,
    )

## Image

Image handling comes in two flavors:

1. **Plain** (`md_plain`) — metadata only (EXIF, dimensions, ...). Without `exiftool` installed this is nearly empty output.
2. **LLM** (`md_llm`) — sends the image to the configured multimodal model. The default prompt asks for a *detailed caption* (`"Write a detailed caption for this image."`); pass `prompt=...` to make it transcribe text instead.

In [4]:
# Plain conversion: no API calls, so image files yield almost no text
convert_and_show(DATA / "images" / "sample_text.jpg")
convert_and_show(DATA / "images" / "sample_image_1.png")

**`data\images\sample_text.jpg`** -> `out\sample_text.md` (0 chars)

---

**`data\images\sample_image_1.png`** -> `out\sample_image_1.md` (0 chars)

---

''

In [5]:
# LLM conversion: one API call per image, transcription prompt, reasoning_effort="none"
for img in [
    "sample_text.jpg",     # simple typed text -> straight transcription
    "sample_image_1.png",  # document-like page (header, table, figure, checklist)
    "page_2.png",          # scanned page from a Japanese engineering document
]:
    convert_and_show(
        DATA / "images" / img,
        converter=md_llm,
        prompt=OCR_PROMPT,
        name=f"llm_{Path(img).stem}.md",
    )

**`data\images\sample_text.jpg`** -> `out\llm_sample_text.md` (65 chars)

---


# Description:
The quick brown fox jumped over the 5 lazy dogs!


**`data\images\sample_image_1.png`** -> `out\llm_sample_image_1.md` (1,409 chars)

---


# Description:
This is the header of the document.

# This is title

## 1. Text

Latin refers to an ancient Italic language originating in the region of Latium in ancient Rome.

## 2. Page Objects

### 2.1 Table

Here's a sample table below, designed to be simple for easy understand and quick reference.

| Name | Corp | Remark |
|---|---|---|
| Foo |  |  |
| Bar | Microsoft | Dummy |

*Table 1: This is a dummy table*

### 2.2. Figure

*Figure 1: Here is a figure with text*

Values: 100, 200, 300, 400, 500
Jan: 200
Feb: 300
Mar: 400
Apr: 450
May: 350
Jun: 250
Month

## 3. Others

AI Document Intelligence is an AI service that applies advanced machine learning to extract text, key-value pairs, tables, and structures from documents automatically and accurately:

- ☑ clear
- ☑ precise
- ☐ vague
- ☑ coherent
- ☐ Incomprehensible

Turn documents into usable data and shift your focus to acting

> ... 509 more characters (full text saved in `out/`)

**`data\images\page_2.png`** -> `out\llm_page_2.md` (1,556 chars)

---


# Description:
[目次へ戻る](#)

| 車両情報　BMC情報 | <u>FRバンパーFR (UPR) + FR CB (UPR)</u> |
|---|---|
| Brand | 部品点数 |
|  | Quantity of Parts |
|  | Total Weight (kg) |
| Cadillac Lyriq Debut Edition 2023 | 1 | 6.84 |
| Smart #1 Premium 2023 | 5 | 13.34 |
| Volkswagen Golf VIII 1.5 eTSI Style 1st Edition II 2020 | 1 | 7.98 |
| Volkswagen Polo 1.0 TSi Highline 2018 | 1 | 7.99 |
| Volkswagen Tiguan 2.0 TDi 4Motion Highline 2016 | 1 | 9.56 |
| Volkswagen Passat Variant 2.0 TDi SCR Highline 2015 | 1 | 7.65 |
| Volkswagen Golf VII 1.4 TSi Comfortline 2013 | 1 | 8.68 |
| Volkswagen Up! 1.0 Take Up! 2012 | 1 | 6.13 |
| Volkswagen Touran 2.0 TDi Highline 2011 | 1 | 7.41 |
| Audi Q4 e-tron 50 quattro 2021 | 1 | 19.31 |
| Hyundai Tucson Plug-in Prime 2021 | 1 | 7.81 |
| Hyundai Ioniq 1.6 Plug-in 2017 | 3 | 6.40 |
| Hyundai i30 1.0 T-GDi Edition One 2016 | 1 | 7.07 |
| Hyundai i10 1.2 Pack Evidence 2014 | 1 |

> ... 656 more characters (full text saved in `out/`)

## PDFs

The built-in PDF converter extracts the **text layer** with `pdfminer` — no API calls, works well for digital PDFs. Converting the whole folder and previewing each result:

In [6]:
# Convert every PDF with the plain text-layer converter, save + preview each
pdf_results = {}
for pdf in sorted((DATA / "pdfs").glob("*.pdf")):
    text = convert_and_show(pdf, name=pdf.stem + ".md", chars=350)
    pdf_results[pdf.name] = text

**`data\pdfs\docling_technical_report.pdf`** -> `out\docling_technical_report.md` (84,201 chars)

---

Docling Technical Report
Version1.0
ChristophAuer MaksymLysak AhmedNassar MicheleDolfi NikolaosLivathinos
PanosVagenas CesarBerrospiRamis MatteoOmenetti FabianLindlbauer
KasperDinkla LokeshMishra YusikKim ShubhamGupta RafaelTeixeiradeLima
ValeryWeber LucasMorin IngmarMeijer ViktorKuropiatnyk PeterW.J.Staar
AI4KGroup,IBMResearch
Ru¨schlikon,Switzerl

> ... 83,851 more characters (full text saved in `out/`)

**`data\pdfs\FinancialStatement-2025-III-BBRI.pdf`** -> `out\FinancialStatement-2025-III-BBRI.md` (123,596 chars)

---

Nomor Surat B.5496-DIR/FMA/10/2025
Nama Emiten PT Bank Rakyat Indonesia (Persero) Tbk
Kode Emiten BBRI
Perihal Penyampaian Laporan Keuangan Interim Yang Tidak Diaudit
Perseroan dengan ini menyampaikan laporan keuangan untuk periode 9 Bulan yang berakhir pada 30/09/2025 dengan ikhtisar sebagai berikut :
Informasi mengenai anak perusahaan Perseroan s

> ... 123,246 more characters (full text saved in `out/`)

**`data\pdfs\long_table_pdf.pdf`** -> `out\long_table_pdf.md` (51,532 chars)

---

...

Design and build
accessible PDF tables
Sample tables
Table 1
| Column header (TH)  | Column header (TH)  | Column header (TH)  |     |
| ------------------- | ------------------- | ------------------- | --- |
| Row header (TH)     | Data cell (TD)      | Data cell (TD)      |     |
| Row header(TH)      | Data cell (TD)      | Data cell (TD)      |

> ... 22,094 more characters (full text saved in `out/`)

**`data\pdfs\Product Strategy & Decision Handbook — Atlas Pay.pdf`** -> `out\Product Strategy & Decision Handbook — Atlas Pay.md` (8,633 chars)

---

Product Strategy & Decision Handbook

Product: AtlasPay

Company: Atlas Fintech Solutions

Last Updated: 12 January 2026

Document Owner: Product Strategy Team

Audience: Product Managers, Product Analysts, Data Analysts, Engineering Leads, Business Stakeholders

1. Purpose of This Document

This handbook serves as the single source of truth for At

> ... 8,283 more characters (full text saved in `out/`)

**`data\pdfs\sustainable_urban_mobility_trends_2025.pdf`** -> `out\sustainable_urban_mobility_trends_2025.md` (3,389 chars)

---

Sustainable Urban
Mobility Trends 2025
This presentation offers urban planners insights into sustainable urban
mobility. We will explore key trends and strategies. Our focus areas include
EV adoption, micromobility, and smart infrastructure. Our goal is to guide
future urban development.
by Muhammad Cikal Merdeka

Key Challenges in Urban Mobility
|

> ... 3,039 more characters (full text saved in `out/`)

In [7]:
# Closer look at a text PDF whose value is almost entirely data tables:
# markitdown has to reconstruct the pipe-table structure from the text layer.
convert_and_show(
    DATA / "pdfs" / "long_table_pdf.pdf",
    name="focus_long_table.md",
    chars=1400,
)

**`data\pdfs\long_table_pdf.pdf`** -> `out\focus_long_table.md` (51,532 chars)

---

REKAP FORMULIR HIMPUNAN PERHITUNGAN LALU LINTAS
(FORMULIR LAPORAN)

SIMPANG KUALA TANJUNG KE INALUM
POS 1 - AKSES ROAD INALUM (PRM TJ GADING)

15-07-2019

Tanggal :
Nama : MOH SUHERI

ANDRIANUS

5a

5b

6a

6b

7a

7b

7c

REKAP FORMULIR HIMPUNAN PERHITUNGAN LALU LINTAS
(FORMULIR LAPORAN)

INALUM ke SIMPANG KUALA TANJUNG
POS 1 - AKSES ROAD INALUM (PRM TJ GADING)

15-07-2019

Tanggal :
Nama : JOKO SAMUDRO

Arah :

Hari 1

Pukul

06.00 - 06.15

06.15 - 06.30

06.30 - 06.45

06.45 - 07.00

07.00 - 07.15

07.15 - 07.30

07.30 - 07.45

07.45 - 08.00

08.00 - 08.15

08.15 - 08.30

08.30 - 08.45

08.45 - 09.00

09.00 - 09.15

09.15 - 09.30

09.30 - 09.45

09.45 - 10.00

10.00 - 10.15

10.15 - 10.30

10.30 - 10.45

10.45 - 11.00

11.00 - 11.15

11.15 - 11.30

11.30 - 11.45

11.45 - 12.00

12.00 - 12.15

12.15 - 12.30

12.30 - 12.45
12.45 - 13.00
13.00 - 13.15

13.15 - 13.30

13.30 - 13.45
13.45 - 14.00

14.00 - 14.15
14.00 - 14.15
14.15 - 14.30
14.30 - 14.45
14.45 - 15.00
15.00 - 15.15
15.15 - 15.30
15.30 - 15.45
15.45 - 16.00

16.00 - 16.15

16.15 - 16.30
16.30 - 16.45
16.45 - 17.00
17.00 - 17.15
17.15 - 17.30
17.30 - 17.45
17.45 - 18.00
18.00 - 18.15
18.15 - 18.30
18.30 - 18.45

18.45 - 19.00
19.00 - 19.15
19.15 - 19.30
19.30 - 19.45
19.45 - 20.00
20.00 - 20.15
20.15 - 20.30
20.30 - 20.45
20.45 - 21.00
21.00 - 21.15
21.15 - 21.30
21.30 - 21.45
21.45 - 22.00
22.00 - 22.15
22.15 - 22.3

> ... 50,132 more characters (full text saved in `out/`)

'REKAP FORMULIR HIMPUNAN PERHITUNGAN LALU LINTAS\n(FORMULIR LAPORAN)\n\nSIMPANG KUALA TANJUNG KE INALUM\nPOS 1 - AKSES ROAD INALUM (PRM TJ GADING)\n\n15-07-2019\n\nTanggal :\nNama : MOH SUHERI\n\nANDRIANUS\n\n5a\n\n5b\n\n6a\n\n6b\n\n7a\n\n7b\n\n7c\n\nREKAP FORMULIR HIMPUNAN PERHITUNGAN LALU LINTAS\n(FORMULIR LAPORAN)\n\nINALUM ke SIMPANG KUALA TANJUNG\nPOS 1 - AKSES ROAD INALUM (PRM TJ GADING)\n\n15-07-2019\n\nTanggal :\nNama : JOKO SAMUDRO\n\nArah :\n\nHari 1\n\nPukul\n\n06.00 - 06.15\n\n06.15 - 06.30\n\n06.30 - 06.45\n\n06.45 - 07.00\n\n07.00 - 07.15\n\n07.15 - 07.30\n\n07.30 - 07.45\n\n07.45 - 08.00\n\n08.00 - 08.15\n\n08.15 - 08.30\n\n08.30 - 08.45\n\n08.45 - 09.00\n\n09.00 - 09.15\n\n09.15 - 09.30\n\n09.30 - 09.45\n\n09.45 - 10.00\n\n10.00 - 10.15\n\n10.15 - 10.30\n\n10.30 - 10.45\n\n10.45 - 11.00\n\n11.00 - 11.15\n\n11.15 - 11.30\n\n11.30 - 11.45\n\n11.45 - 12.00\n\n12.00 - 12.15\n\n12.15 - 12.30\n\n12.30 - 12.45\n12.45 - 13.00\n13.00 - 13.15\n\n13.15 - 13.30\n\n13.30 - 13.45\n13

### Scanned PDF → LLM (vision OCR)

For a **scanned** PDF the text layer is missing, so the plain conversion returns (almost) nothing. Pipeline instead:

1. **Render** pages to PNG with `pymupdf` (raise `dpi=` for small print).
2. **Transcribe** each page image with the multimodal model (`gpt-6-luna`, `reasoning_effort="none"`), using the shared `OCR_PROMPT`.
3. **Stitch** the per-page markdown into one document.

> Note: all demo PDFs here have a text layer; `sustainable_urban_mobility_trends_2025.pdf` is the most image-heavy (a Gamma slide export), so it makes a realistic demo target — but the same pipeline works for true scans.
>
> The `markitdown-ocr` plugin (subsections below) does the equivalent through `md.convert()` itself - its PDF converter even detects text-free pages and OCRs a 300 DPI render automatically.

In [8]:
# 1) Render the PDF's pages to PNG
SCAN_PDF = DATA / "pdfs" / "sustainable_urban_mobility_trends_2025.pdf"
PAGES_DIR = OUT / "scan_pages"
PAGES_DIR.mkdir(exist_ok=True)

doc = pymupdf.open(SCAN_PDF)
rendered = []
for i, page in enumerate(doc, start=1):
    png = PAGES_DIR / f"{SCAN_PDF.stem}_p{i:02d}.png"
    page.get_pixmap(dpi=150).save(png)
    rendered.append(png)
doc.close()
print(f"rendered {len(rendered)} pages -> {PAGES_DIR}")

rendered 8 pages -> out\scan_pages


In [9]:
# 2) Transcribe each rendered page with the multimodal model (direct API call,
#    reasoning_effort="none"), then 3) stitch everything into one document.
def vision_to_markdown(image_path, prompt=OCR_PROMPT, *, model=MODEL, effort=REASONING_EFFORT):
    b64 = base64.b64encode(Path(image_path).read_bytes()).decode("utf-8")
    ext = Path(image_path).suffix.lstrip(".").lower()
    mime = "jpeg" if ext == "jpg" else ext
    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {"type": "image_url", "image_url": {"url": f"data:image/{mime};base64,{b64}"}},
                ],
            }
        ],
        extra_body={"reasoning_effort": effort},
    )
    return response.choices[0].message.content or ""


pages_md = []
for png in rendered:
    print("ocr:", png.name, "...", end=" ")
    t0 = time.perf_counter()
    text = vision_to_markdown(png).strip()
    print(f"{len(text):,} chars in {time.perf_counter() - t0:.1f}s")
    pages_md.append(f"<!-- page {png.stem[-2:]} -->\n\n{text}")

scanned_md = "\n\n---\n\n".join(pages_md)
scanned_path = save_markdown("scanned_" + SCAN_PDF.stem + ".md", scanned_md)
SUMMARY.append((str(SCAN_PDF), "llm-scanned", len(scanned_md)))
display(Markdown(f"**`{SCAN_PDF.name}`** -> `{scanned_path}` ({len(scanned_md):,} chars)\n\n---"))
display(Markdown(preview(scanned_md, 1200)))

ocr: sustainable_urban_mobility_trends_2025_p01.png ... 337 chars in 3.8s
ocr: sustainable_urban_mobility_trends_2025_p02.png ... 479 chars in 2.1s
ocr: sustainable_urban_mobility_trends_2025_p03.png ... 470 chars in 3.8s
ocr: sustainable_urban_mobility_trends_2025_p04.png ... 420 chars in 3.3s
ocr: sustainable_urban_mobility_trends_2025_p05.png ... 472 chars in 3.0s
ocr: sustainable_urban_mobility_trends_2025_p06.png ... 491 chars in 3.2s
ocr: sustainable_urban_mobility_trends_2025_p07.png ... 477 chars in 2.4s
ocr: sustainable_urban_mobility_trends_2025_p08.png ... 314 chars in 2.5s


**`sustainable_urban_mobility_trends_2025.pdf`** -> `out\scanned_sustainable_urban_mobility_trends_2025.md` (3,653 chars)

---

<!-- page 01 -->

# Sustainable Urban
# Mobility Trends 2025

This presentation offers urban planners insights into sustainable urban mobility. We will explore key trends and strategies. Our focus areas include EV adoption, micromobility, and smart infrastructure. Our goal is to guide future urban development.

by Muhammad Cikal Merdeka

Made with GAMMA

---

<!-- page 02 -->

# Key Challenges in Urban Mobility

## Traffic Congestion

US cities lose $166 billion annually. Traffic impacts productivity and quality of life.

## Air Pollution

Transportation causes 27% of US GHG emissions. This harms public health.

## Limited Accessibility

Marginalized communities often lack transit options. This creates social inequity.

## Aging Infrastructure

A $4.5 trillion investment is needed by 2025. Modernization is critical for efficiency.

Made with GAMMA

---

<!-- page 03 -->

# Emerging Technologies

## Electric Vehicles (EVs)

Global market projected at $800 billion by 2027. EVs reduce tailpipe emissions.

## Micromobility

E-scooters and e-bikes cut congestion by 10%. They offer flexible urban travel.

## Smart Infrastructure

Sensors and data optimize traffic flow. This improves effi

> ... 2,453 more characters (full text saved in `out/`)

### markitdown-ocr plugin (PDF)

With `enable_plugins=True` in the setup cell, the plugin's PDF converter takes over `md.convert()` — it OCRs embedded images inline in reading order and wraps results as `*[Image OCR] … [End OCR]*`. Pages without a text layer are auto-rendered at 300 DPI and sent whole to the vision model.

The mobility PDF holds **14 unique embedded images** → that many vision calls per run (repeated bytes are recognized once). Prefer `docling technical report` for diagram-heavy output (~30 calls) by changing the path below.

In [ ]:
# ~14 vision calls (14 unique embedded images; repeated bytes are recognized once).
convert_and_show(
    DATA / "pdfs" / "sustainable_urban_mobility_trends_2025.pdf",
    converter=md_ocr,
    name="ocr_sustainable_urban_mobility_trends_2025.md",
    chars=1200,
)

## PPT

Slide decks convert to Markdown with one section per slide. With an `llm_client` configured, embedded pictures additionally get an LLM-generated description — that mode is optional below since it costs one call per picture.

In [10]:
pptx = DATA / "powerpoints" / "sustainable_urban_mobility_trends_2025.pptx"
convert_and_show(pptx, name=pptx.stem + ".md", chars=1500)

**`data\powerpoints\sustainable_urban_mobility_trends_2025.pptx`** -> `out\sustainable_urban_mobility_trends_2025.md` (4,228 chars)

---

<!-- Slide number: 1 -->

![preencoded.png](Image0.jpg)
Sustainable Urban Mobility Trends 2025
This presentation offers urban planners insights into sustainable urban mobility. We will explore key trends and strategies. Our focus areas include EV adoption, micromobility, and smart infrastructure. Our goal is to guide future urban development.
by Muhammad Cikal Merdeka

![preencoded.png](Image1.jpg)

<!-- Slide number: 2 -->
Key Challenges in Urban Mobility
Traffic Congestion
Air Pollution
Limited Accessibility
Aging Infrastructure
Transportation causes 27% of US GHG emissions. This harms public health.
US cities lose $166 billion annually. Traffic impacts productivity and quality of life.
Marginalized communities often lack transit options. This creates social inequity.
A $4.5 trillion investment is needed by 2025. Modernization is critical for efficiency.

<!-- Slide number: 3 -->

![preencoded.png](Image0.jpg)
Emerging Technologies

![preencoded.png](Image1.jpg)

![preencoded.png](Image2.jpg)
Electric Vehicles (EVs)
Micromobility
E-scooters and e-bikes cut congestion by 10%. They offer flexible urban travel.
Global market projected at $800 billion by 2027. EVs reduce tailpipe emissions.

![preencoded.png](Image3.jpg)

![preencoded.png](Image4.jpg)
Smart Infrastructure
MaaS (Mobility-as-a-Service)
Sensors and data optimize traffic flow. This improves efficiency and safety.
Integrated platforms reduce car usage by 20%. They streamline transportation access.

<!-- Slide number

> ... 2,728 more characters (full text saved in `out/`)

"<!-- Slide number: 1 -->\n\n![preencoded.png](Image0.jpg)\nSustainable Urban Mobility Trends 2025\nThis presentation offers urban planners insights into sustainable urban mobility. We will explore key trends and strategies. Our focus areas include EV adoption, micromobility, and smart infrastructure. Our goal is to guide future urban development.\nby Muhammad Cikal Merdeka\n\n![preencoded.png](Image1.jpg)\n\n<!-- Slide number: 2 -->\nKey Challenges in Urban Mobility\nTraffic Congestion\nAir Pollution\nLimited Accessibility\nAging Infrastructure\nTransportation causes 27% of US GHG emissions. This harms public health.\nUS cities lose $166 billion annually. Traffic impacts productivity and quality of life.\nMarginalized communities often lack transit options. This creates social inequity.\nA $4.5 trillion investment is needed by 2025. Modernization is critical for efficiency.\n\n<!-- Slide number: 3 -->\n\n![preencoded.png](Image0.jpg)\nEmerging Technologies\n\n![preencoded.png](Image1.

In [11]:
# Optional: same deck, but also describe slide images with the LLM
RUN_PPTX_LLM = False  # set True to spend extra API calls on slide imagery
if RUN_PPTX_LLM:
    convert_and_show(pptx, converter=md_llm, prompt=OCR_PROMPT, name=f"llm_{pptx.stem}.md", chars=1200)

### markitdown-ocr plugin (PPT)

The plugin's PPTX converter handles picture shapes, image placeholders, and grouped images, asking the LLM for a description first and using OCR as the fallback when no description is returned. Core shape ordering, tables, charts, and speaker notes are preserved.

With **30 unique images** this deck is the costliest target — deliberately behind a flag.

In [ ]:
RUN_PPTX_PLUGIN_OCR = False  # ~30 vision calls on this deck
if RUN_PPTX_PLUGIN_OCR:
    convert_and_show(
        pptx,
        converter=md_ocr,
        name=f"ocr_{pptx.stem}.md",
        chars=1400,
    )

## Excel

Workbooks come out as Markdown, with one pipe-separated table per worksheet — handy for feeding tabular data to LLMs.

In [12]:
xlsx = DATA / "spreadsheets" / "example_table.xlsx"
convert_and_show(xlsx, name=xlsx.stem + ".md", chars=1600)

**`data\spreadsheets\example_table.xlsx`** -> `out\example_table.md` (21,071 chars)

---

## Sheet1
| Unnamed: 0 | Code | Amount Raw | Amount Processed | Order ID | Month |
| --- | --- | --- | --- | --- | --- |
| 0 | 5.502600e+11 | 64000 | 62720.00 | TRD2022122305306 | January |
| 1 | 5.502600e+11 | 58092 | 56930.16 | TRD2022122500888 | January |
| 2 | 5.502600e+11 | 43062 | 42200.76 | TRD2022122500830 | January |
| 3 | 5.502600e+11 | 489000 | 479220.00 | TRD2022122606284 | January |
| 4 | 5.502600e+11 | 10000 | 9800.00 | TRD2022122701823 | January |
| 5 | 5.502600e+11 | 24000 | 23520.00 | TRD2022122706644 | January |
| 6 | 5.502600e+11 | 24000 | 23520.00 | TRD2022122706674 | January |
| 7 | 5.502600e+11 | 48080 | 47118.40 | TRD2022122801939 | January |
| 8 | 5.502600e+11 | 149274 | 146288.52 | TRD2022122802089 | January |
| 9 | 5.502600e+11 | 110000 | 107800.00 | TRD2022122802390 | January |
| 10 | 5.502600e+11 | 34000 | 33320.00 | TRD2022122907187 | January |
| 11 | 5.502600e+11 | 116200 | 113876.00 | TRD2022122801897 | January |
| 12 | 5.502600e+11 | 271510 | 266079.80 | TRD2022123004626 | January |
| 13 | 5.502600e+11 | 110000 | 107800.00 | TRD2023010204762 | January |
| 14 | 5.502600e+11 | 44000 | 43120.00 | TRD2023010206759 | January |
| 15 | 5.502600e+11 | 111000 | 108780.00 | TRD2023010301543 | January |
| 16 | 5.502600e+11 | 50000 | 49000.00 | TRD2023010301562 | January |
| 17 | 5.502600e+11 | 59000 | 57820.00 | TRD2023010308689 | January |
| 18 | 5.502600e+11 | 36000 | 35280.00 | TRD2023010409180 | January |
| 19 | 5.502600e+11 | 64000 | 62720.00 | TRD2023010510977 | January |
| 20 | 5.502600e+11 | 140000 | 137200.00 | TRD2023010701418 | January |
| 21

> ... 19,471 more characters (full text saved in `out/`)

'## Sheet1\n| Unnamed: 0 | Code | Amount Raw | Amount Processed | Order ID | Month |\n| --- | --- | --- | --- | --- | --- |\n| 0 | 5.502600e+11 | 64000 | 62720.00 | TRD2022122305306 | January |\n| 1 | 5.502600e+11 | 58092 | 56930.16 | TRD2022122500888 | January |\n| 2 | 5.502600e+11 | 43062 | 42200.76 | TRD2022122500830 | January |\n| 3 | 5.502600e+11 | 489000 | 479220.00 | TRD2022122606284 | January |\n| 4 | 5.502600e+11 | 10000 | 9800.00 | TRD2022122701823 | January |\n| 5 | 5.502600e+11 | 24000 | 23520.00 | TRD2022122706644 | January |\n| 6 | 5.502600e+11 | 24000 | 23520.00 | TRD2022122706674 | January |\n| 7 | 5.502600e+11 | 48080 | 47118.40 | TRD2022122801939 | January |\n| 8 | 5.502600e+11 | 149274 | 146288.52 | TRD2022122802089 | January |\n| 9 | 5.502600e+11 | 110000 | 107800.00 | TRD2022122802390 | January |\n| 10 | 5.502600e+11 | 34000 | 33320.00 | TRD2022122907187 | January |\n| 11 | 5.502600e+11 | 116200 | 113876.00 | TRD2022122801897 | January |\n| 12 | 5.502600e+11 | 2715

In [13]:
# Same output in raw form: see the real pipe-table syntax markitdown produced
print((OUT / (xlsx.stem + ".md")).read_text(encoding="utf-8")[:2000])

## Sheet1
| Unnamed: 0 | Code | Amount Raw | Amount Processed | Order ID | Month |
| --- | --- | --- | --- | --- | --- |
| 0 | 5.502600e+11 | 64000 | 62720.00 | TRD2022122305306 | January |
| 1 | 5.502600e+11 | 58092 | 56930.16 | TRD2022122500888 | January |
| 2 | 5.502600e+11 | 43062 | 42200.76 | TRD2022122500830 | January |
| 3 | 5.502600e+11 | 489000 | 479220.00 | TRD2022122606284 | January |
| 4 | 5.502600e+11 | 10000 | 9800.00 | TRD2022122701823 | January |
| 5 | 5.502600e+11 | 24000 | 23520.00 | TRD2022122706644 | January |
| 6 | 5.502600e+11 | 24000 | 23520.00 | TRD2022122706674 | January |
| 7 | 5.502600e+11 | 48080 | 47118.40 | TRD2022122801939 | January |
| 8 | 5.502600e+11 | 149274 | 146288.52 | TRD2022122802089 | January |
| 9 | 5.502600e+11 | 110000 | 107800.00 | TRD2022122802390 | January |
| 10 | 5.502600e+11 | 34000 | 33320.00 | TRD2022122907187 | January |
| 11 | 5.502600e+11 | 116200 | 113876.00 | TRD2022122801897 | January |
| 12 | 5.502600e+11 | 271510 | 266079.80 |

### markitdown-ocr plugin (XLSX)

Workbook images are OCR'd from the repaired workbook and listed under a `### Images in this sheet:` heading after each sheet's table. `example_table.xlsx` has no images today — expect table-only output identical to plain; designed here so you can opt in when an image-bearing workbook arrives. Legacy `.xls` files stay on the core converter, without image OCR.

In [ ]:
RUN_XLSX_PLUGIN_OCR = False  # free today (0 images); opt in for image-bearing versions
if RUN_XLSX_PLUGIN_OCR:
    convert_and_show(
        xlsx,
        converter=md_ocr,
        name=f"ocr_{xlsx.stem}.md",
        chars=1600,
    )

## Recap

What each section produced — source, converter used, and markdown size. Full outputs are in `out/`.

In [14]:
# Recap table
if SUMMARY:
    seen = set()
    rows = ["| source | converter | md chars |", "|---|---|---:|"]
    for source, kind, n in SUMMARY:
        key = (Path(source).name, kind)
        if key in seen:
            continue
        seen.add(key)
        rows.append(f"| `{Path(source).name}` | {kind} | {n:,} |")
    display(Markdown("\n".join(rows)))

| source | converter | md chars |
|---|---|---:|
| `sample_text.jpg` | plain | 0 |
| `sample_image_1.png` | plain | 0 |
| `sample_text.jpg` | llm | 65 |
| `sample_image_1.png` | llm | 1,409 |
| `page_2.png` | llm | 1,556 |
| `docling_technical_report.pdf` | plain | 84,201 |
| `FinancialStatement-2025-III-BBRI.pdf` | plain | 123,596 |
| `long_table_pdf.pdf` | plain | 51,532 |
| `pdf_with_tables_example.pdf` | plain | 22,444 |
| `Product Strategy & Decision Handbook — Atlas Pay.pdf` | plain | 8,633 |
| `sustainable_urban_mobility_trends_2025.pdf` | plain | 3,389 |
| `sustainable_urban_mobility_trends_2025.pdf` | llm-scanned | 3,653 |
| `sustainable_urban_mobility_trends_2025.pptx` | plain | 4,228 |
| `example_table.xlsx` | plain | 21,071 |